# Traceability — Dev Log

## Objetivo e papel no pipeline

`core/traceability` agrupa eventos de auditoria já gravados em
`core/audit_logs` (V1) numa cadeia de proveniência consultável — "quais
eventos pertencem à mesma operação de negócio?".

**Escopo honesto**: não é tracing distribuído estilo OpenTelemetry (nenhum
`trace_id` é propagado automaticamente entre módulos nesta versão) — é
correlação por uma chave de payload que os eventos já compartilham (ex.
`project_name`, presente em todos os eventos que o `ripd_engine` grava para
um mesmo RIPD).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import tempfile
from pathlib import Path
from core.audit_logs.logger import AuditLogger
from core.traceability.tracer import trace_by_correlation_key
from shared.schemas import AuditEventType

demo_dir = Path(tempfile.mkdtemp(prefix="traceability_demo_"))
logger = AuditLogger(log_path=demo_dir / "audit_log.jsonl")

# Simula os eventos que um pipeline real (ripd_engine) geraria para o mesmo projeto.
logger.record_event(AuditEventType.PII_SCAN, actor="ripd_engine", payload={"project_name": "Chatbot FAQ", "step": "pii_scan"})
logger.record_event(AuditEventType.POLICY_EVALUATION, actor="ripd_engine", payload={"project_name": "Chatbot FAQ", "step": "policy_eval"})
logger.record_event(AuditEventType.RAG_QUERY, actor="ripd_engine", payload={"project_name": "Chatbot FAQ", "step": "rag_query"})
logger.record_event(AuditEventType.RIPD_GENERATED, actor="ripd_engine", payload={"project_name": "Chatbot FAQ", "step": "ripd_generated"})
logger.record_event(AuditEventType.PII_SCAN, actor="ripd_engine", payload={"project_name": "Outro Projeto", "step": "pii_scan"})

traces = trace_by_correlation_key("project_name", logger=logger)
print(f"{len(traces)} trace(s) de proveniência construídos (um por project_name distinto):")
for t in traces:
    print(f"  trace_id={t.trace_id[:8]}... | {len(t.event_ids)} evento(s)")
    print(f"  {t.summary}")

2 trace(s) de proveniência construídos (um por project_name distinto):
  trace_id=2fc8d080... | 4 evento(s)
  4 evento(s) correlacionado(s) por 'project_name'='Chatbot FAQ' (pii_scan, policy_evaluation, rag_query, ripd_generated), entre 2026-08-20T23:37:21.528916+00:00 e 2026-08-20T23:37:21.541132+00:00.
  trace_id=b93df646... | 1 evento(s)
  1 evento(s) correlacionado(s) por 'project_name'='Outro Projeto' (pii_scan), entre 2026-08-20T23:37:21.546470+00:00 e 2026-08-20T23:37:21.546470+00:00.


## Rodando a suíte de testes

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/traceability/tests -v
```

7 testes sobre `AuditLogger` real isolada em arquivo temporário: agrupamento
correto; erro sem match; resumo menciona contagem/tipos; um trace por valor
distinto; eventos sem a chave são ignorados; log vazio; ordenação
cronológica.

## Handoff Summary

- **Status:** ✅ done — 7/7 testes passando.
- **TODO onda futura:** gravação automática de um `trace_id` correlato pelo
  próprio `ripd_engine`/`governance_copilot` em todos os eventos de uma
  mesma operação, eliminando a dependência de escolher a chave de
  correlação certa manualmente.